# Création d'applications de génération d'images (OpenAI)

Les LLM ne se limitent pas à la génération de texte. Vous pouvez aussi générer des images à partir de descriptions textuelles. Les images en tant que modalité sont utiles dans la technologie médicale, l'architecture, le tourisme, le développement de jeux, le marketing, et plus encore. Dans cette leçon, nous travaillons avec les modèles **GPT Image** actuels sur la plateforme OpenAI.

## Objectifs d'apprentissage

À la fin de cette leçon, vous serez capable de :

- Expliquer ce qu'est la génération d'images et où elle est utile.
- Comprendre la famille de modèles `gpt-image` et en quoi elle diffère des modèles DALL·E hérités.
- Construire une application de génération d’images et éditer des images.

## Qu'est-ce que la génération d'images ?

Les modèles de génération d’images créent des images à partir d’un texte de commande. Les modèles modernes comme `gpt-image` apprennent la relation entre le texte et les images lors de leur entraînement, puis transforment itérativement un bruit aléatoire en une image correspondant à votre commande.

Deux familles bien connues de modèles d’image sont :

- **`gpt-image` (OpenAI)** — la génération actuelle utilisée dans cette leçon. Il supporte la génération d’image à partir du texte et l’édition d’images (retouche avec masque).
- **Midjourney** — un modèle tiers populaire avec son propre service et un flux de travail basé sur Discord.

> Les anciens modèles d’image OpenAI — **DALL·E 2** et **DALL·E 3** — sont hérités. DALL·E 3 n’est plus disponible pour les nouveaux déploiements, et des fonctionnalités comme `create_variation` n’existaient que dans DALL·E 2. Utilisez les modèles `gpt-image` pour les nouvelles applications.

> **Important :** Les modèles `gpt-image` retournent l’image générée au format **base64** (`b64_json`), pas sous forme d’URL. Votre code décode la chaîne base64 en octets et la sauvegarde — il n’y a pas d’URL d’image à télécharger.


## Créer votre première application de génération d’images

Alors, que faut-il pour créer une application de génération d’images ? Vous avez besoin des bibliothèques suivantes :

- **python-dotenv**, il est fortement recommandé d’utiliser cette bibliothèque pour garder vos secrets dans un fichier *.env* séparé du code.
- **openai**, cette bibliothèque est ce que vous utiliserez pour interagir avec l’API OpenAI.
- **pillow**, pour travailler avec des images en Python.


1. Créez un fichier *.env* avec le contenu suivant :

    ```text
    OPENAI_API_KEY='<add your OpenAI key here>'
    ```



1. Rassemblez les bibliothèques ci-dessus dans un fichier appelé *requirements.txt* comme suit :

    ```text
    python-dotenv
    openai
    pillow
    ```


1. Ensuite, créez un environnement virtuel et installez les bibliothèques :


In [ ]:
# create virtual env
! python3 -m venv venv
# activate environment
! source venv/bin/activate
# install libraries
# pip install -r requirements.txt, if using a requirements.txt file 
! pip install python-dotenv openai pillow

> [!NOTE]
> For Windows, use the following commands to create and activate your virtual environment:

    ```bash
    python3 -m venv venv
    venv\Scripts\activate.bat
    ````

1. Add the following code in file called *app.py*:

    ```python
    import os
    import base64
    from PIL import Image
    import dotenv
    import openai

    # import dotenv
    dotenv.load_dotenv()

    # Create OpenAI object (reads OPENAI_API_KEY from your .env)
    client = openai.OpenAI()


    try:
        # Create an image by using the image generation API
        generation_response = client.images.generate(
            model="gpt-image-2",
            prompt='Bunny on horse, holding a lollipop, on a foggy meadow where it grows daffodils',    # Enter your prompt text here
            size='1024x1024',
            n=1
        )
        # Set the directory for the stored image
        image_dir = os.path.join(os.curdir, 'images')

        # If the directory doesn't exist, create it
        if not os.path.isdir(image_dir):
            os.mkdir(image_dir)

        # Initialize the image path (note the filetype should be png)
        image_path = os.path.join(image_dir, 'generated-image.png')

        # gpt-image models return the image as base64 (b64_json), not a URL
        image_b64 = generation_response.data[0].b64_json
        generated_image = base64.b64decode(image_b64)
        with open(image_path, "wb") as image_file:
            image_file.write(generated_image)

        # Display the image in the default image viewer
        image = Image.open(image_path)
        image.show()

    # catch exceptions
    except openai.BadRequestError as err:
        print(err)

    ```

Let's explain this code:

- First, we import the libraries we need, including the OpenAI library, the dotenv library, the base64 module, and the Pillow library.

    ```python
    import os
    import base64
    from PIL import Image
    import dotenv
    import openai
    ```

- After that, we create the client, which reads the API key from your ``.env``.

    ```python
    # Create OpenAI object
    client = openai.OpenAI()
    ```

- Next, we generate the image:

    ```python
    generation_response = client.images.generate(
        model="gpt-image-2",
        prompt='Bunny on horse, holding a lollipop, on a foggy meadow where it grows daffodils',    # Enter your prompt text here
        size='1024x1024',
        n=1
    )
    ```

    `gpt-image` models return the image as a **base64** string in `data[0].b64_json`. We decode it to bytes and write it to a file — there's no URL to download.

    ```python
    image_b64 = generation_response.data[0].b64_json
    generated_image = base64.b64decode(image_b64)
    with open(image_path, "wb") as image_file:
        image_file.write(generated_image)
    ```

- Lastly, we open the image and use the standard image viewer to display it:

    ```python
    image = Image.open(image_path)
    image.show()
    ```

### More details on generating the image

Let's look at the parameters of `images.generate`:

- **model**, is the image model to use (for example `gpt-image-2`).
- **prompt**, is the text prompt used to generate the image. Here it's "Bunny on horse, holding a lollipop, on a foggy meadow where it grows daffodils".
- **size**, is the size of the generated image (for example `1024x1024`, `1536x1024`, `1024x1536`, or `"auto"`).
- **n**, is the number of images generated. Here we generate one.

> Image models don't take a `temperature` parameter — that's a text-generation control. To get variety, call the API again; to reduce variety, make your prompt more specific.

## Additional capabilities of image generation

You've seen how to generate an image with a few lines of Python. `gpt-image` models can also **edit** an existing image. By providing an image, an optional **mask** (which marks the area to change), and a prompt, you can alter part of an image — for example, add a hat to our bunny.

```python
response = client.images.edit(
  model="gpt-image-2",
  image=open("base_image.png", "rb"),
  mask=open("mask.png", "rb"),
  prompt="An image of a rabbit with a hat on its head.",
  n=1,
  size="1024x1024"
)
# edits are also returned as base64
edited_image = base64.b64decode(response.data[0].b64_json)
```

The base image contains just the rabbit; the final image has the hat on the rabbit.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Avertissement** :
Ce document a été traduit à l'aide du service de traduction automatique [Co-op Translator](https://github.com/Azure/co-op-translator). Bien que nous nous efforçions d'assurer l'exactitude, veuillez noter que les traductions automatisées peuvent contenir des erreurs ou des inexactitudes. Le document original dans sa langue native doit être considéré comme la source faisant autorité. Pour les informations critiques, il est recommandé de recourir à une traduction professionnelle réalisée par un humain. Nous ne saurions être tenus responsables des malentendus ou erreurs d'interprétation découlant de l'utilisation de cette traduction.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
